# Day 24 In-Class Assignment: S3 Data Ingest & Reproducible Project Structure

**Use this notebook to guide your in-class work on Day 24.**

You will work to:
- Create and configure an S3 bucket
- Upload NYC 311 Service Request dataset files with intentional folder structure
- Initialize a Git repository with professional project structure
- Create/customize a data dictionary and README
- Document your S3 paths and dataset assumptions

If you get stuck, **check in with a classmate or ask for help.**

**At the end of class, your repo will be pushed to GitHub, and this notebook will be submitted to D2L.**

> **Name (FIRST and LAST):** _delete this text and type your name here before submitting this notebook_

---

## Learning objectives

By the end of this class period, you should be able to:

- **Create an S3 bucket** with appropriate security and versioning settings.
- **Understand S3 prefixes** (pseudo-folders) and how to structure data for reproducibility.
- **Explain data provenance** (source, assumptions, limitations) and why it matters for teamwork and future-you.
- **Initialize a Git repository** with professional structure: `sql/`, `notes/`, `reports/`, README, data dictionary.
- **Document your dataset** (columns, types, units) in a data dictionary that supports SQL queries.

---

## Table of contents

1. [NYC 311 dataset orientation](#dataset)
2. [AWS S3 bucket setup](#s3)
3. [Local project structure and Git](#git)
4. [S3 data upload](#upload)
5. [Documentation (README + data dictionary)](#docs)
6. [Commit and push](#commit)
7. [Wrap-up and reflection](#wrapup)

- - -

<a id="dataset"></a>

## 1. NYC 311 dataset orientation

### What is NYC 311?

NYC 311 is the city's non-emergency service request system. Citizens call, text, or submit online to report issues: potholes, noise complaints, missed garbage pickups, etc. Today's dataset contains a **sample of 200,000 requests from Q1 2026** across 15 city agencies.

**Two main tables:**
- **`complaints.csv`** (200k rows): contains things like identification number, creation date, agency, problem type, borough, location, resolution status, etc.
- **`agencies.csv`** (~15 rows): contains agency codes and agency names.

### Connecting to the Day 23 problem brief

Though we were short on time, the "problem brief" activity we looked at during the Day 23 class period was designed to help you connect a hypothetical dataset to a real-world stakeholder scenario. The goals for this activity were to:
- Convert a stakeholder scenario into a data question that can be answered with the dataset
- Identify which columns in the dataset are relevant to that question
- Determine metrics for success (what does a "good" answer look like?)
- Consider assumptions and limitations of the dataset for answering that question.

This sort of thinking should help to guide your work today onwards. When you upload the dataset to S3, create your project structure, and document your data dictionary, you should be thinking about how a future analyst (maybe future-you) would understand and use this dataset to answer a question like the one you developed in the problem brief.

### Downloading and unzipping

If you haven't already:

1. [Download `nyc_311.zip` from the course website](https://msu-cmse-courses.github.io/cmse492-moderntools-ss26/_static/downloads/day_24/nyc_311.zip).
2. Unzip to an appropriate directory of your choosing (you'll be uploading from here).
3. You should see:
   - `complaints.csv`
   - `agencies.csv`
   - `DATA_DICTIONARY.md`
   - `README_template.md`

- - -

<a id="s3"></a>

## 2. AWS S3 bucket setup

### Why S3?

S3 (stands for Simple Storage Service) is **object storage**: "cheap" (relatively), durable, scalable. It's the foundation for enterprise data workflows. Today you'll practice the basics.

As a class, **we went through the process of creating an S3 bucket together**. The checklist below is a reference for you to use if you were absent, weren't able to follow, or need to create another bucket in the future.

**You can skip this if your bucket is already set up from class.**

### Bucket creation checklist

**Step 1: Log in to AWS Learner Lab**
- Open your Learner Lab but logging into the AWS Academy Course (refer to Day 23 content as needed).
- Start your lab and wait for it to initialize.
- Confirm it says "Running" and shows the countdown timer.

**Step 2: Create an S3 bucket**

Navigate: AWS Console → S3 → "Create bucket"

Use a logical naming scheme, e.g., `cmse492-<your-msu-netid>-nyc311` or `cmse492-<firstname>-nyc311`

**Step 3: Bucket settings**

Make sure these are set:

| Setting | Value | Why |
| :-- | :-- | :-- |
| **Region** | `us-east-1` (it should default to this) | Must match your Learner Lab region |
| **Bucket namespace** | Account Regional namespace | Avoids naming conflicts with other buckets in global namespace |
| **Object ownership** | ACLs disabled (Bucket owner enforced) | Simplifies permissions management |
| **Versioning** | Enable | Lets you recover old versions if you overwrite by mistake |
| **Public access** | Block | Keep course data private |
| **Tags** | `owner:yourname`, `course:cmse492`, `project:nyc311` | Easy tracking and cleanup |
| **Encryption** | S3-managed keys (default) | Secure at rest |

- - -

<a id="git"></a>

## 3. Local project structure and Git

For this part of the assignment, you'll be working on setting up a Git repository to document and track your work on this cloud-based "mini-project". 

### Initialize the repo locally

**On your **laptop**:**

1. Create your project folder using the following naming scheme: `aws-nyc311-yourMSUNetID` (make sure to replace `yourMSUNetID` with your actual MSU NetID!)
2. Initialize the Git repository with `git init`

### Create project structure

While still in your project folder, create the following subdirectories:

- `sql` (for SQL scripts)
- `notes` (for meeting notes and planning)
- `reports` (for generated reports)

### Verify project structure

Make sure everything looks correct by listing the contents of your project directory:

```bash
ls -la # (On Windows PowerShell: dir)
```

You should see:

```
.git/
sql/
notes/
reports/
```


### Copy documentation templates

Now you want to make sure you have the documentation templates in place. 

From your **unzipped** `nyc_311.zip`, copy or move and rename the following files into your project root directory (`aws-nyc311-yourMSUNetID`):
- `README_template.md` → `README.md` (make sure to drop the "_template" part of the name so it's just `README.md`)
- `DATA_DICTIONARY.md` → `DATA_DICTIONARY.md`

### Connect to GitHub

Create an empty **private** repo on GitHub.com (no README, no .gitignore, no license). Make sure to use the same naming scheme as your local folder, i.e., `aws-nyc311-yourMSUNetID`.

Then in your terminal:

```bash
# Set your GitHub remote
git remote add origin https://github.com/YOUR_USERNAME/aws-nyc311-yourMSUNetID.git

# Verify
git remote -v
# Should show: origin https://github.com/YOUR_USERNAME/...
```

- - -

<a id="upload"></a>

## 4. S3 data upload

### Upload with proper structure

**Goal:** Upload CSVs to `raw/` prefix so the S3 path looks like `s3://your-bucket/raw/complaints.csv`.

**Steps:**

1. **Important:** We don't upload to the bucket root. Instead:
   - In the S3 console, within your bucket, create a folder named `raw` so that you can upload CSVs *into* that folder
2. Then: S3 Console → Your bucket name → Click "Upload"
3. Select files:
   - Find `complaints.csv` and `agencies.csv` (from your unzipped folder)
   - Click "Upload" (you shouldn't need to change any of the default settings in the upload interface)

**Result:**

Once you've uploaded, your S3 bucket should have this structure:

```
s3://cmse492-<your-msu-netid>-nyc311/ # Or whatever your bucket name is
  └── raw/
      ├── complaints.csv
      └── agencies.csv
```

### Verify upload

In S3 console:
1. Click on the `raw/` folder
2. Click on `agencies.csv` → Click "Open" to preview/download it to confirm it worked (it's small; do NOT open/download `complaints.csv` for now given its size)

### Why specify "raw/" prefix?

Using a `raw/` prefix (pseudo-folder) helps to:
- Organize data by stage (raw, processed, etc.)
- Avoid cluttering the bucket root
- Make it clear which files are the original source data

**Important note:** One important professional practice when working with data and thinking about data provenance is to always keep the original source data separate and clearly labeled. This way, you can always go back to the "raw" data if you need to reprocess it or check assumptions, and it prevents accidental overwriting of the original files.

### S3 paths for documentation

Write down these paths (you'll put them in README):

> **S3 paths (insert your bucket name)**
> 
> **Bucket root:**  
> s3://___
> 
> **Complaints data:**  
> s3://___/raw/complaints.csv
> 
> **Agencies data:**  
> s3://___/raw/agencies.csv

- - -

<a id="docs"></a>

## 5. Documentation (README + data dictionary)

### Edit README.md

Now that you've uploaded your data, it's time to document it. The README is the first place a future analyst (or stakeholder) will look to understand what this project is about, where the data came from, and how to use it.

Open the `README.md` file (copied from template). Then, update:

1. **S3 paths**
   - Find and replace placeholders with your S3 bucket name and confirm the paths are correct

2. **Description** (2–3 sentences on what the project does)
   - Review the template content from the README and determine where it makes the most sense to put your project description.
   - Then, put in some sort of placeholder text that you can update later with a more detailed description.
      - Example: "Analysis of NYC 311 service requests to identify patterns in complaint types by agency and borough."

3. **Assumptions and known issues**
   - Review this section and, based on your understanding of the dataset at this time, add any assumptions you think are relevant.

### Review data dictionary

The `DATA_DICTIONARY.md` file describes the data and each column for both files. This is a critical resource for anyone who wants to understand or use the dataset, especially for writing SQL queries later on.

**Review and note:**

> **5.2 Data dictionary review (mark with "X" to indicate completion)**
> 
> - [ ] I've read through the data dictionary
> - [ ] At least 3 columns make sense to me
> - [ ] I've identified the primary key (likely `unique_key`)
> - [ ] I've noted any columns that seem ambiguous
> 
> **One ambiguous column or question:**  
> _If any, write here; otherwise write "none":_

- - -

<a id="commit"></a>

## 6. Commit and push

### Stage and commit locally

Once you've completed all of the previous sections, make sure to stage and commit your changes with a clear message (modify the example message as appropriate):

```bash
git add . # Confirm that you're not accidentally adding any large files (like the CSVs) that shouldn't be in Git, create a .gitignore if needed
git commit -m "Initial NYC 311 project structure with S3 paths and data dictionary"
```

**Note:**: You may need to add .gitkeep files in your folders to ensure they are tracked by Git, since Git doesn't track empty folders. You can create an empty `.gitkeep` file in each folder (e.g., `sql/.gitkeep`, `notes/.gitkeep`, `reports/.gitkeep`) to keep the structure.

### Push to GitHub

```bash
git push -u origin main
```

### Verify on GitHub

Go to your repo on GitHub.com. You should see:

```
├── README.md
├── DATA_DICTIONARY.md
├── sql/ (folder)
├── notes/ (folder)
└── reports/ (folder)
```

- - -

<a id="wrapup"></a>

## 7. Wrap-up and reflection

Nice work. You've completed the infrastructure for your AWS mini-project.

### What you've built

- **S3 bucket** with properly versioned, organized data
- **Local Git repo** with professional structure
- **Documentation** (README + data dictionary) explaining data provenance

### Reflection

If you have time, take a moment to reflect on today's work and how it sets you up for success in the next phases of this project.

> **7.1 What was the most challenging part of today's setup?**  
> _Your answer:_
>
> **7.2 What feels solid, and what do you still have questions about?**  
> _Your answer:_
>
> **7.3 How will this S3/Git structure help you when you revisit this project in 6 months?**  
> _Your answer (think about documentation, reproducibility, etc.):_

- - -

### Congratulations, you're done!

Submit this assignment by uploading it to the course Desire2Learn web page. Go to the In-class assignments section, find the appropriate submission link, and upload it there.

See you in class!

---
© Copyright 2026, The Department of Computational Mathematics, Science and Engineering at Michigan State University.